# LSH Applications: Three End-to-End Case Studies

## Learning Objectives

By the end of this notebook you will be able to:
- Apply the full LSH pipeline to near-duplicate document detection, entity resolution, and code plagiarism detection
- Choose the right shingling strategy and LSH family for each problem
- Measure precision, recall, and speedup relative to brute-force comparison
- Reason about practical trade-offs: memory, FPR/FNR, threshold selection

## Application 1: Near-Duplicate Web Page Detection

**Problem.** Given a corpus of web pages, find all near-duplicate pairs with Jaccard similarity J ≥ 0.8.

**Why this matters.** Search engines crawl billions of pages and must de-duplicate before indexing. A page copied with minor changes (mirror sites, scrapers, reformatted content) wastes storage and distorts ranking.

**Pipeline.**
1. Convert each page to character 9-shingles
2. Hash each shingle to an integer ID (to save memory)
3. Compute MinHash signatures of length k=200
4. Apply LSH with b=50 bands of r=4 rows → threshold t ≈ (1/b)^(1/r) = (1/50)^(1/4) ≈ 0.376 (conservative — catches pairs well below 0.8)
5. Verify every candidate pair with exact Jaccard

The conservative threshold means we'll see false positives (pairs flagged but J<0.8) — that's fine, because verification is cheap.

In [1]:
import numpy as np
import time
import random
import hashlib
from itertools import combinations
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------
# Core utilities shared across all three applications
# -----------------------------------------------------------------------

def shingle_chars(text, k=9):
    """Character k-shingles of text (as a set of ints via hashing)."""
    return set(
        int(hashlib.md5(text[i:i+k].encode()).hexdigest(), 16) % (2**32)
        for i in range(len(text) - k + 1)
    )

def shingle_words(text, k=2):
    "Word k-shingles (bigrams by default) as set of ints."
    tokens = text.lower().split()
    return set(
        int(hashlib.md5(' '.join(tokens[i:i+k]).encode()).hexdigest(), 16) % (2**32)
        for i in range(len(tokens) - k + 1)
    )

def minhash_signatures(shingle_sets, n_hashes=200, rng=None):
    """
    Compute MinHash signatures using linear hash functions h(x) = (a*x + b) mod p mod m.
    Returns (n_docs, n_hashes) array of int32.
    """
    if rng is None:
        rng = np.random.default_rng(42)
    n_docs = len(shingle_sets)
    # large prime
    p = (1 << 31) - 1
    m = p
    a = rng.integers(1, p, size=n_hashes)
    b = rng.integers(0, p, size=n_hashes)

    sigs = np.full((n_docs, n_hashes), np.iinfo(np.int32).max, dtype=np.int64)

    for doc_idx, shingles in enumerate(shingle_sets):
        for shingle in shingles:
            hashes = (a * int(shingle) + b) % p
            sigs[doc_idx] = np.minimum(sigs[doc_idx], hashes)

    return sigs.astype(np.int32)

def lsh_candidate_pairs(signatures, b, r):
    """
    Band-based LSH candidate generation.

    Parameters
    ----------
    signatures : (n_docs, k) int array where k = b * r
    b          : number of bands
    r          : rows per band

    Returns
    -------
    set of (i, j) pairs with i < j
    """
    n_docs, k = signatures.shape
    assert k == b * r, f"k={k} != b*r={b*r}"
    candidates = set()
    for band in range(b):
        bucket_dict = {}
        band_sigs = signatures[:, band * r : (band + 1) * r]
        for doc_idx in range(n_docs):
            key = tuple(band_sigs[doc_idx])
            if key not in bucket_dict:
                bucket_dict[key] = []
            bucket_dict[key].append(doc_idx)
        for doc_list in bucket_dict.values():
            if len(doc_list) > 1:
                for i, j in combinations(doc_list, 2):
                    candidates.add((min(i, j), max(i, j)))
    return candidates

def jaccard_exact(set_a, set_b):
    """Exact Jaccard similarity between two sets."""
    ""
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union > 0 else 0.0

print("Utilities loaded.")


Utilities loaded.


In [2]:
# -----------------------------------------------------------------------
# Application 1: Near-Duplicate Web Page Detection
# -----------------------------------------------------------------------

rng_app1 = np.random.default_rng(123)
random.seed(42)

# --- Synthetic corpus generation ---

def random_text(n_words, vocab_size=200, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    words = [f"word{i}" for i in range(vocab_size)]
    return ' '.join(rng.choice(words, size=n_words).tolist())

def near_duplicate(base_text, modify_frac=0.15, rng=None):
    """Replace modify_frac of words with random alternatives."""
    ""
    if rng is None:
        rng = np.random.default_rng()
    words = base_text.split()
    n_replace = max(1, int(len(words) * modify_frac))
    indices = rng.choice(len(words), size=n_replace, replace=False)
    for idx in indices:
        new_word_id = rng.integers(0, 200)
        words[idx] = f"word{new_word_id}"
    return ' '.join(words)

# Generate 100 base documents, 40 near-duplicates, 20 unrelated
n_base = 100
n_near_dup = 40
n_unrelated = 20

base_docs   = [random_text(300, rng=rng_app1) for _ in range(n_base)]
dup_docs    = [near_duplicate(base_docs[i % n_base], modify_frac=rng_app1.uniform(0.05, 0.20), rng=rng_app1)
               for i in range(n_near_dup)]
other_docs  = [random_text(300, rng=rng_app1) for _ in range(n_unrelated)]

corpus = base_docs + dup_docs + other_docs
doc_labels = ['base']*n_base + ['dup']*n_near_dup + ['other']*n_unrelated
n_docs = len(corpus)
print(f"Corpus: {n_docs} documents ({n_base} base, {n_near_dup} near-dupes, {n_unrelated} unrelated)")

# --- Build ground truth: pairs with J >= 0.8 ---
shingle_sets = [shingle_chars(doc, k=9) for doc in corpus]

t0 = time.time()
ground_truth = set()
for i, j in combinations(range(n_docs), 2):
    if jaccard_exact(shingle_sets[i], shingle_sets[j]) >= 0.80:
        ground_truth.add((i, j))
brute_time = time.time() - t0
print(f"Ground truth (J≥0.80): {len(ground_truth)} pairs  [{brute_time:.2f}s brute force]")

# --- LSH pipeline ---
k = 200
b, r = 50, 4
print(f"\nMinHash signatures: k={k}, LSH: b={b} bands × r={r} rows")
print(f"S-curve threshold: t ≈ {(1/b)**(1/r):.3f}")

t1 = time.time()
sigs = minhash_signatures(shingle_sets, n_hashes=k, rng=rng_app1)
lsh_time_sig = time.time() - t1

t2 = time.time()
candidates = lsh_candidate_pairs(sigs, b=b, r=r)
lsh_time_band = time.time() - t2

# --- Verify candidates ---
verified_dupes = set()
for i, j in candidates:
    if jaccard_exact(shingle_sets[i], shingle_sets[j]) >= 0.80:
        verified_dupes.add((i, j))

total_pairs = n_docs * (n_docs - 1) // 2
precision = len(verified_dupes) / len(candidates) if candidates else 0
recall    = len(verified_dupes) / len(ground_truth) if ground_truth else 0
speedup   = total_pairs / len(candidates) if candidates else float('inf')

print(f"\n--- Results ---")
print(f"Total possible pairs:  {total_pairs:,}")
print(f"Candidates (LSH):      {len(candidates):,}  ({100*len(candidates)/total_pairs:.1f}% of all pairs)")
print(f"True duplicates found: {len(verified_dupes)}")
print(f"Ground truth size:     {len(ground_truth)}")
print(f"Precision:             {precision:.3f}")
print(f"Recall:                {recall:.3f}")
print(f"Speedup vs brute force: {speedup:.1f}x fewer comparisons")
print(f"Signature time: {lsh_time_sig:.3f}s  |  Band time: {lsh_time_band:.3f}s")


Corpus: 160 documents (100 base, 40 near-dupes, 20 unrelated)


Ground truth (J≥0.80): 20 pairs  [1.63s brute force]

MinHash signatures: k=200, LSH: b=50 bands × r=4 rows
S-curve threshold: t ≈ 0.376



--- Results ---
Total possible pairs:  12,720
Candidates (LSH):      4,009  (31.5% of all pairs)
True duplicates found: 20
Ground truth size:     20
Precision:             0.005
Recall:                1.000
Speedup vs brute force: 3.2x fewer comparisons
Signature time: 0.589s  |  Band time: 0.007s


## Application 2: Entity Resolution / Record Linkage

**Problem.** Two databases contain company names. The same company appears under different spellings, abbreviations, and punctuation. Find matching entities without brute-force comparing every pair across databases.

**Why LSH?** With two databases of 1M companies each, brute-force would require 10¹² comparisons. LSH reduces this to O(n) by hashing each company name into bands and only comparing names that share a band bucket.

**Approach.** Use word 2-shingles on normalized names:
- Lowercase, strip punctuation, split into tokens
- 2-shingles capture word bigrams (overlapping pairs of adjacent tokens)
- "acme corp" → {"acme corp"} and "acme corporation" → {"acme corp", "corp oration"} share shingles

**Key insight.** Even though "Corp" ≠ "Corporation", their 2-shingles do overlap because the abbreviation is embedded in the full form. Character shingles would work even better for typos, but word shingles handle abbreviations more gracefully.

In [3]:
import re

# -----------------------------------------------------------------------
# Application 2: Entity Resolution
# -----------------------------------------------------------------------

def normalize_company(name):
    """Lowercase, strip punctuation, expand common abbreviations."""
    name = name.lower()
    # common abbreviations
    abbrevs = {
        r'\bcorp\.?\b': 'corporation',
        r'\bco\.?\b': 'company',
        r'\bltd\.?\b': 'limited',
        r'\binc\.?\b': 'incorporated',
        r'\bllc\b': 'llc',
    }
    for pat, rep in abbrevs.items():
        name = re.sub(pat, rep, name)
    # remove non-alphanumeric except spaces
    name = re.sub(r'[^a-z0-9 ]', ' ', name)
    return ' '.join(name.split())

def company_shingles(name, k=2):
    """Word k-shingles for a company name."""
    ""
    return shingle_words(normalize_company(name), k=k)

# --- Generate synthetic company name pairs ---
base_companies = [
    ("Acme Corp", ["ACME Corporation", "Acme Co.", "acme corp", "ACME CORP."]),
    ("General Electric", ["General Electric Co.", "GE Corp", "general electric company"]),
    ("International Business Machines", ["IBM Corp.", "I.B.M. Corporation", "International Business Machines Corp"]),
    ("Microsoft Corporation", ["Microsoft Corp.", "MICROSOFT INC", "Microsoft Co"]),
    ("Apple Inc", ["Apple Computer Inc.", "APPLE INCORPORATED", "Apple Inc."]),
    ("Goldman Sachs", ["Goldman Sachs Group Inc.", "Goldman Sachs & Co", "Goldman Sachs Co."]),
    ("JPMorgan Chase", ["JP Morgan Chase & Co.", "JPMorgan Chase Bank", "J.P. Morgan Chase"]),
    ("Procter Gamble", ["Procter & Gamble Co.", "P&G Corp", "Procter and Gamble Company"]),
    ("Johnson Johnson", ["Johnson & Johnson Corp.", "J&J Inc.", "Johnson and Johnson"]),
    ("Exxon Mobil", ["ExxonMobil Corporation", "Exxon Mobil Corp.", "Exxon Mobile Inc"]),
]

# Build two databases: db_a has canonical names, db_b has variants
db_a = []
db_b = []
ground_truth_pairs = set()

for base_idx, (canonical, variants) in enumerate(base_companies):
    a_idx = len(db_a)
    db_a.append(canonical)
    for variant in variants:
        b_idx = len(db_b)
        db_b.append(variant)
        ground_truth_pairs.add((a_idx, b_idx))

# Add some unrelated noise companies to each database
noise_a = ["Zenith Industries", "Plex Analytics", "Nova Systems", "Orion Solutions",
           "Apex Technologies", "Delta Group", "Sigma Labs", "Theta Capital"]
noise_b = ["Pinnacle Corp", "Summit Industries", "Crest Solutions", "Ridge Analytics",
           "Peak Technologies", "Vortex Group", "Cascade Labs", "Horizon Capital"]
db_a.extend(noise_a)
db_b.extend(noise_b)

print(f"Database A: {len(db_a)} companies")
print(f"Database B: {len(db_b)} companies")
print(f"True matching pairs: {len(ground_truth_pairs)}")
print(f"Total possible comparisons (brute force): {len(db_a) * len(db_b)}")

# --- Compute shingles and signatures ---
shingles_a = [company_shingles(name, k=2) for name in db_a]
shingles_b = [company_shingles(name, k=2) for name in db_b]

# Add offset to db_b indices so we can run LSH on combined set
combined_shingles = shingles_a + shingles_b
offset_b = len(db_a)

rng_app2 = np.random.default_rng(77)
k_sig = 100
b_sig, r_sig = 20, 5
sigs_combined = minhash_signatures(combined_shingles, n_hashes=k_sig, rng=rng_app2)
candidates_all = lsh_candidate_pairs(sigs_combined, b=b_sig, r=r_sig)

# Keep only cross-database candidates (one index < offset_b, one >= offset_b)
cross_candidates = set()
for i, j in candidates_all:
    if i < offset_b <= j:
        cross_candidates.add((i, j - offset_b))
    elif j < offset_b <= i:
        cross_candidates.add((j, i - offset_b))

# Verify candidates
verified = set()
for a_idx, b_idx in cross_candidates:
    j = jaccard_exact(shingles_a[a_idx], shingles_b[b_idx])
    if j >= 0.1:   # low threshold: entity resolution needs high recall
        verified.add((a_idx, b_idx))

precision = len([p for p in verified if p in ground_truth_pairs]) / len(verified) if verified else 0
recall    = len([p for p in verified if p in ground_truth_pairs]) / len(ground_truth_pairs)
speedup   = (len(db_a) * len(db_b)) / len(cross_candidates) if cross_candidates else float('inf')

print(f"\n--- Entity Resolution Results ---")
print(f"Candidates (cross-db):  {len(cross_candidates)}")
print(f"Verified matches:       {len(verified)}")
print(f"True matches in result: {len([p for p in verified if p in ground_truth_pairs])}")
print(f"Precision: {precision:.3f}  |  Recall: {recall:.3f}")
print(f"Speedup vs brute force: {speedup:.1f}x")

# Show some found pairs
print("\nSample found pairs:")
for a_idx, b_idx in sorted(verified)[:8]:
    j = jaccard_exact(shingles_a[a_idx], shingles_b[b_idx])
    marker = "✓" if (a_idx, b_idx) in ground_truth_pairs else "✗"
    print(f"  {marker}  '{db_a[a_idx]}' ~ '{db_b[b_idx]}'  (J={j:.3f})")


Database A: 18 companies
Database B: 39 companies
True matching pairs: 31
Total possible comparisons (brute force): 702

--- Entity Resolution Results ---
Candidates (cross-db):  9
Verified matches:       9
True matches in result: 9
Precision: 1.000  |  Recall: 0.290
Speedup vs brute force: 78.0x

Sample found pairs:
  ✓  'Acme Corp' ~ 'ACME Corporation'  (J=1.000)
  ✓  'Acme Corp' ~ 'acme corp'  (J=1.000)
  ✓  'Acme Corp' ~ 'ACME CORP.'  (J=1.000)
  ✓  'International Business Machines' ~ 'International Business Machines Corp'  (J=0.667)
  ✓  'Microsoft Corporation' ~ 'Microsoft Corp.'  (J=1.000)
  ✓  'Apple Inc' ~ 'APPLE INCORPORATED'  (J=1.000)
  ✓  'Apple Inc' ~ 'Apple Inc.'  (J=1.000)
  ✓  'JPMorgan Chase' ~ 'JPMorgan Chase Bank'  (J=0.500)


## Application 3: Duplicate Code Detection (Fingerprinting)

**Problem.** Detect copied or plagiarised Python code files. Common scenarios:
- Student submits another student's code with variable renames
- Open-source code copied into a codebase with minor modifications
- Code fragments reused across projects

**Approach.**
- Tokenize Python code using the `tokenize` module (handles keywords, identifiers, operators)
- For robustness to variable renames: normalize identifier names to generic tokens (VAR_0, VAR_1, ...)
- Use token 4-shingles (sequences of 4 consecutive tokens)
- Run MinHash + LSH to find pairs with high shingle overlap

**Why token shingles?**
- Character shingles would flag whitespace/comment differences as false negatives
- Token shingles abstract away formatting
- 4-grams are long enough to represent code structure (a loop, an expression, a function call)

In [4]:
import tokenize
import io
import keyword

# -----------------------------------------------------------------------
# Application 3: Code Plagiarism Detection
# -----------------------------------------------------------------------

def tokenize_python(source):
    """
    Tokenize Python source code, returning a list of (type, string) pairs.
    Skips comments, newlines, encoding markers, and end-of-file tokens.
    """
    tokens = []
    try:
        reader = io.StringIO(source).readline
        for tok in tokenize.generate_tokens(reader):
            if tok.type in (tokenize.COMMENT, tokenize.NEWLINE,
                            tokenize.NL, tokenize.ENCODING,
                            tokenize.ENDMARKER):
                continue
            tokens.append((tok.type, tok.string))
    except tokenize.TokenError:
        pass
    return tokens

def normalize_tokens(token_list):
    """
    Replace user-defined identifier names with generic VAR_0, VAR_1, ...
    Keywords, operators, and literals are kept as-is.
    """
    name_map = {}
    normalized = []
    for tok_type, tok_str in token_list:
        if tok_type == tokenize.NAME and not keyword.iskeyword(tok_str):
            # user-defined identifier
            if tok_str not in name_map:
                name_map[tok_str] = f"VAR_{len(name_map)}"
            normalized.append(name_map[tok_str])
        else:
            normalized.append(tok_str)
    return normalized

def code_shingles(source, k=4):
    """Token k-shingles of Python source code."""
    ""
    tokens = normalize_tokens(tokenize_python(source))
    return set(
        int(hashlib.md5(' '.join(tokens[i:i+k]).encode()).hexdigest(), 16) % (2**32)
        for i in range(len(tokens) - k + 1)
    )

# --- Synthetic code snippets ---
code_base = [
    # snippet 0: bubble sort
    '''
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr
''',
    # snippet 1: binary search
    '''
def binary_search(arr, target):
    low, high = 0, len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1
''',
    # snippet 2: fibonacci
    '''
def fibonacci(n):
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b
''',
    # snippet 3: matrix multiply
    '''
def matrix_multiply(A, B):
    rows_A = len(A)
    cols_A = len(A[0])
    cols_B = len(B[0])
    result = [[0] * cols_B for _ in range(rows_A)]
    for i in range(rows_A):
        for j in range(cols_B):
            for k in range(cols_A):
                result[i][j] += A[i][k] * B[k][j]
    return result
''',
]

# Create copies with variable renames
code_copies = [
    # copy of bubble_sort with renamed variables
    '''
def sort_array(data):
    size = len(data)
    for idx in range(size):
        for pos in range(0, size - idx - 1):
            if data[pos] > data[pos + 1]:
                data[pos], data[pos + 1] = data[pos + 1], data[pos]
    return data
''',
    # copy of binary_search with renamed variables
    '''
def find_element(nums, value):
    left, right = 0, len(nums) - 1
    while left <= right:
        center = (left + right) // 2
        if nums[center] == value:
            return center
        elif nums[center] < value:
            left = center + 1
        else:
            right = center - 1
    return -1
''',
    # near-duplicate fibonacci (extra comment line)
    '''
def fib_sequence(n):
    if n <= 1:
        return n
    prev, curr = 0, 1
    for step in range(2, n + 1):
        prev, curr = curr, prev + curr
    return curr
''',
]

# Unrelated snippets
code_other = [
    '''
def count_words(text):
    words = text.lower().split()
    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0) + 1
    return counts
''',
    '''
class Stack:
    def __init__(self):
        self.items = []
    def push(self, item):
        self.items.append(item)
    def pop(self):
        return self.items.pop() if self.items else None
''',
]

all_code = code_base + code_copies + code_other
code_labels = (
    [f'base_{i}' for i in range(len(code_base))] +
    [f'copy_{i}' for i in range(len(code_copies))] +
    [f'other_{i}' for i in range(len(code_other))]
)

# Ground truth: base_i ~ copy_i for i in {0,1,2}
ground_truth_code = {(i, len(code_base) + i) for i in range(len(code_copies))}

# --- Compute shingles and signatures ---
code_shingle_sets = [code_shingles(src, k=4) for src in all_code]

print("Token counts after normalization:")
for i, src in enumerate(all_code):
    toks = normalize_tokens(tokenize_python(src))
    print(f"  {code_labels[i]:12s}: {len(toks)} tokens, {len(code_shingle_sets[i])} 4-shingles")

rng_app3 = np.random.default_rng(55)
k_code = 60
b_code, r_code = 15, 4
sigs_code = minhash_signatures(code_shingle_sets, n_hashes=k_code, rng=rng_app3)
candidates_code = lsh_candidate_pairs(sigs_code, b=b_code, r=r_code)

print(f"\nLSH candidates: {len(candidates_code)} pairs")
print("All candidate pairs and their exact Jaccard:")
for i, j in sorted(candidates_code):
    j_val = jaccard_exact(code_shingle_sets[i], code_shingle_sets[j])
    gt_marker = "TRUE MATCH" if (i, j) in ground_truth_code or (j, i) in ground_truth_code else ""
    print(f"  ({code_labels[i]}, {code_labels[j]})  J={j_val:.3f}  {gt_marker}")

found = sum(1 for i,j in candidates_code if (i,j) in ground_truth_code or (j,i) in ground_truth_code)
print(f"\nGround truth matches detected: {found} / {len(ground_truth_code)}")


Token counts after normalization:
  base_0      : 80 tokens, 67 4-shingles
  base_1      : 77 tokens, 73 4-shingles
  base_2      : 49 tokens, 45 4-shingles
  base_3      : 105 tokens, 96 4-shingles
  copy_0      : 80 tokens, 67 4-shingles
  copy_1      : 77 tokens, 73 4-shingles
  copy_2      : 49 tokens, 45 4-shingles
  other_0     : 47 tokens, 44 4-shingles
  other_1     : 59 tokens, 50 4-shingles

LSH candidates: 3 pairs
All candidate pairs and their exact Jaccard:
  (base_0, copy_0)  J=1.000  TRUE MATCH
  (base_1, copy_1)  J=1.000  TRUE MATCH
  (base_2, copy_2)  J=1.000  TRUE MATCH

Ground truth matches detected: 3 / 3


In [5]:
# -----------------------------------------------------------------------
# Performance Comparison: LSH vs Brute Force
# -----------------------------------------------------------------------

import time

def run_brute_force(shingle_sets, threshold=0.5):
    """Find all pairs with J >= threshold by brute force."""
    n = len(shingle_sets)
    results = set()
    for i, j in combinations(range(n), 2):
        if jaccard_exact(shingle_sets[i], shingle_sets[j]) >= threshold:
            results.add((i, j))
    return results

def run_lsh_pipeline(shingle_sets, k=100, b=20, r=5, threshold=0.5):
    """Full MinHash + LSH pipeline."""
    ""
    rng_local = np.random.default_rng(1)
    sigs = minhash_signatures(shingle_sets, n_hashes=k, rng=rng_local)
    candidates = lsh_candidate_pairs(sigs, b=b, r=r)
    # verify candidates
    verified = set()
    for ci, cj in candidates:
        if jaccard_exact(shingle_sets[ci], shingle_sets[cj]) >= threshold:
            verified.add((ci, cj))
    return candidates, verified

# Generate corpora of increasing size
n_values = [50, 100, 150, 200, 300, 400, 500]
threshold = 0.5
k, b, r = 100, 20, 5

print(f"{'n':>6} {'BF_time':>10} {'LSH_time':>10} {'BF_pairs':>10} "
      f"{'candidates':>12} {'cands/n²':>10} {'precision':>10} {'recall':>8}")
print("-" * 90)

rng_perf = np.random.default_rng(999)

for n in n_values:
    # generate corpus: 80% original, 20% near-duplicates
    n_orig = int(0.8 * n)
    n_dup  = n - n_orig
    base_texts = [random_text(200, rng=rng_perf) for _ in range(n_orig)]
    dup_texts  = [near_duplicate(base_texts[i % n_orig], rng=rng_perf) for i in range(n_dup)]
    corpus_n = base_texts + dup_texts
    shingles_n = [shingle_chars(doc, k=9) for doc in corpus_n]

    # brute force
    t_bf_start = time.time()
    bf_results = run_brute_force(shingles_n, threshold=threshold)
    bf_time = time.time() - t_bf_start

    # LSH
    t_lsh_start = time.time()
    candidates_n, lsh_results = run_lsh_pipeline(shingles_n, k=k, b=b, r=r, threshold=threshold)
    lsh_time = time.time() - t_lsh_start

    total_pairs = n * (n - 1) // 2
    prec = len(lsh_results & bf_results) / len(lsh_results) if lsh_results else 0
    rec  = len(lsh_results & bf_results) / len(bf_results) if bf_results else 1.0
    cands_frac = len(candidates_n) / total_pairs

    print(f"{n:>6} {bf_time:>10.3f} {lsh_time:>10.3f} {len(bf_results):>10} "
          f"{len(candidates_n):>12} {cands_frac:>10.4f} {prec:>10.3f} {rec:>8.3f}")

print("\nNote: LSH time includes signature computation + band hashing + candidate verification.")
print("Brute force time scales O(n²); LSH time scales roughly O(n).")


     n    BF_time   LSH_time   BF_pairs   candidates   cands/n²  precision   recall
------------------------------------------------------------------------------------------


    50      0.083      0.113         10           17     0.0139      1.000    1.000


   100      0.331      0.224         20           65     0.0131      1.000    1.000


   150      0.750      0.336         30          171     0.0153      1.000    1.000


   200      2.183      0.457         40          282     0.0142      1.000    1.000


   300      3.062      0.658         60          509     0.0113      1.000    1.000


   400      5.458      0.937         80         1043     0.0131      1.000    0.988


   500      8.641      1.256        100         1749     0.0140      1.000    0.980

Note: LSH time includes signature computation + band hashing + candidate verification.
Brute force time scales O(n²); LSH time scales roughly O(n).


## Practical Considerations

### Memory

For a MinHash pipeline with k=200 hashes and int32 values:
- Per document: 200 × 4 bytes = **800 bytes/document**
- 1 million documents: **800 MB** for signatures (fits in a modern server's RAM)
- 1 billion documents: 800 GB — requires distributed storage (e.g. sharded by band)

Character shingles are much larger (potentially millions of unique shingles per corpus), but you never need to store them — only the MinHash signature.

### Choosing the Threshold t

Start from the **business requirement**, not from algorithm defaults:
1. What Jaccard similarity J* means for your problem (e.g. J* = 0.8 for web pages, J* = 0.5 for news articles)
2. Set b and r so the S-curve threshold is slightly *below* J* (to favor recall over precision)
3. Use exact-Jaccard verification to filter false positives from candidates

Rule of thumb: target FNR ≤ 0.05 (miss at most 5% of true pairs); FPR can be higher since verification is cheap.

### False Negatives vs False Positives

| Error Type | Cost | Strategy |
|---|---|---|
| False negative | **Permanent miss** — you will never re-examine this pair | Keep FNR low (< 5%) by using more bands |
| False positive | **Cheap to fix** — verify with exact Jaccard | Allow higher FPR; pay O(1) per candidate to verify |

**Consequence:** in practice, use a *lower* threshold for band hashing than your true business threshold J*. Let verification filter the false positives.

### Production Engineering Notes

- **Columnar signature storage.** Lay out signatures as one column per band (not one row per document) for cache-friendly band lookups.
- **Batch band hashing.** Hash all n document signatures for band b together before moving to band b+1.
- **Approximate counting for web scale.** At the scale of the entire web (billions of pages), even storing candidate pairs is too expensive. Use streaming near-duplicate detection with count-min sketches (see `ml_005_04_ams_sketch_algorithm.ipynb`).
- **Incremental updates.** When new documents arrive, compute their signatures and probe each band's hash table — no need to reprocess existing documents.